# Practice 55 — pandas + NumPy


In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Museum visitor survey

`pd.crosstab(index, columns)` builds a frequency table — counts how often each combination of two categorical variables appears. The result is a DataFrame.

```python
pd.crosstab(df['a'], df['b'])                      # raw counts
pd.crosstab(df['a'], df['b'], normalize='index')   # row proportions (each row sums to 1.0)
pd.crosstab(df['a'], df['b'], normalize='columns') # column proportions
```

- Build a crosstab of `exhibit` vs `rating` (raw counts).
- Normalize by row. Which exhibit has the highest proportion of 5-star ratings? Use `np.argmax`.
- Among members only, what is the mean rating per exhibit? Use `groupby`.
- Use `np.percentile` to find the 25th and 75th percentile of all ratings.


In [ ]:
np.random.seed(3)
n = 300
visitors = pd.DataFrame(
    {
        "exhibit": np.random.choice(
            ["Ancient Egypt", "Space", "Natural History", "Modern Art"], n
        ),
        "age_group": np.random.choice(["Under 18", "18–34", "35–54", "55+"], n),
        "rating": np.random.choice([1, 2, 3, 4, 5], n),
        "member": np.random.choice([True, False], n, p=[0.3, 0.7]),
    }
)

# Your code here
pd.crosstab(visitors["exhibit"], visitors["rating"])

pdf = pd.crosstab(visitors["exhibit"], visitors["rating"], normalize="index")
pdf.index[np.argmax(pdf[5])]

visitors.loc[visitors["member"]].groupby("exhibit")["rating"].mean()

np.percentile(visitors["rating"], [25, 75])

array([2., 4.])

---

## Level 2 — Monthly product revenue

- Stack into long format `(month, category)` and name the Series `revenue`.
- Use `shift(12, freq='MS')` + `join` to align last year's data. Compute `yoy_pct` per category (rounded to 1 decimal).
- Use `np.corrcoef` to find the correlation between `electronics` and `clothing` monthly revenue.
- Which category had the highest mean YoY growth? Use `np.nanmean` across categories and `np.argmax` to identify it.


In [40]:
np.random.seed(17)
idx = pd.date_range("2023-01-01", periods=18, freq="MS")
revenue = pd.DataFrame(
    {
        "electronics": np.random.randint(120, 200, 18),
        "clothing": np.random.randint(80, 140, 18),
        "home": np.random.randint(50, 100, 18),
    },
    index=idx,
)
revenue.index.name = "month"

# Your code here

rs = revenue.stack().rename_axis(['month','category'])

last = revenue.shift(12, freq = 'MS')

al = revenue.join(last, rsuffix='_last')


In [41]:
for c in revenue.columns:
    ch = al[c] - al[c+'_last']
    pct = round(ch/al[c + '_last']*100,1)
    al[c+'_yoy'] = pct

In [44]:
np.corrcoef(al['electronics'],al['clothing'])[0,1]

np.float64(0.05689862121441456)

In [53]:
# y = {}
# for c in revenue.columns:
#     cy = al[c +'_yoy']
#     y[c] = np.nanmean(cy)
# max(y,key = y.get)

y = pd.Series({c: np.nanmean(al[c+'_yoy']) for c in revenue.columns})
y.idxmax()

'home'

---

## Level 3 — Employee performance

1. Convert `dept` and `level` to `category`. Make `level` ordered: Junior < Mid < Senior < Lead.
2. Use `pd.crosstab` to show promotion counts by `dept` (rows) vs `level` (columns). Normalize by column.
3. Add `score_tier`: `'High'` if score ≥ 80, `'Mid'` if score ≥ 65, else `'Low'`. Use `.apply()`.
4. For each dept, compute the correlation between `tenure_yrs` and `score` using `np.corrcoef`. Collect results in a dict.
5. Which dept–level combo has the highest mean score? Use `groupby` with a named agg, then `.idxmax()`.


In [64]:
np.random.seed(99)
n = 250
employees = pd.DataFrame(
    {
        "dept": np.random.choice(["Engineering", "Sales", "Marketing", "Support"], n),
        "level": np.random.choice(["Junior", "Mid", "Senior", "Lead"], n),
        "tenure_yrs": np.round(np.random.exponential(3, n), 1),
        "score": np.random.randint(50, 100, n),
        "promoted": np.random.choice([0, 1], n, p=[0.75, 0.25]),
    }
)

# Your code here

ol = pd.CategoricalDtype(['Junior','Mid','Senior'], ordered=True)
employees['level'] = employees['level'].astype(ol)
employees['dept'] = employees['dept'].astype('category')

pd.crosstab(employees.loc[employees['promoted']==1,'dept'], employees.loc[employees['promoted']==1,'level'], normalize='columns')

employees['score_tier'] = pd.cut(employees['score'], bins = [0,65,80,np.inf], labels = ['Low','Mid','High'], right=False)
{d:np.corrcoef(employees.loc[employees['dept']==d,'tenure_yrs'],employees.loc[employees['dept']==d,'score'])[0,1] for d in employees['dept'].unique()}
employees.groupby(['dept','level'], observed=True)['score'].agg('mean').idxmax()

('Marketing', 'Junior')